Вырезаем объекты из фотографий коко

In [ ]:
import json

In [ ]:
# Открываем файл для чтения
with open(r"instances_val2014.json", "r") as f:
    data = json.load(f)  # загружаем JSON в Python

In [ ]:
# Посмотреть все ключи верхнего уровня
print(data.keys())  

# Пример: посмотреть первые 2 изображения
print(data["images"][:2])

# Пример: посмотреть первые 2 аннотации
print(data["annotations"][:2])

In [ ]:
!pip install pycocotools

In [ ]:
import cv2
import numpy as np

def is_connected(mask):
    """
    Проверяет, состоит ли объект из одной связной области.
    mask: бинарная маска (np.uint8, 0/255)
    Возвращает True, если объект связный, False если несколько частей
    """
    # Находим контуры
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # Если контуров больше одного → объект несвязный
    return len(contours) == 1

После выполнения следующего блока в заданной папке появится 80 папок для каждого класса и в каждой папке будут лежать вырезанные объекты

In [ ]:
import json
import cv2
import os
import numpy as np

# ------------------------ Настройки ------------------------
json_file = r"instances_val2014.json"
images_folder = r"val2014"  # папка с фото
output_folder = r"output"   # папка для сохранения классов
os.makedirs(output_folder, exist_ok=True)

# Создаем список всех классов COCO
target_classes = [cat["name"] for cat in coco["categories"]] # заданы все 80 классов
MIN_SIZE = 250

# ------------------------ Загрузка JSON ------------------------
with open(json_file, "r") as f:
    coco = json.load(f)

categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
image_id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}

# счётчики
class_counters = {cls: 0 for cls in target_classes}

# создаём папки для классов
for cls in target_classes:
    os.makedirs(os.path.join(output_folder, cls), exist_ok=True)

# ------------------------ Основной цикл ------------------------
for img_id, file_name in image_id_to_file.items():
    img_path = os.path.join(images_folder, file_name)
    img = cv2.imread(img_path)

    if img is None:
        print(f"Не удалось открыть {file_name}")
        continue
        
    
    
    anns = [ann for ann in coco["annotations"] if ann["image_id"] == img_id]

    for ann in anns:
        category_id = ann["category_id"]
        class_name = categories[category_id]
        
        iscrowd = ann.get("iscrowd", 0)
        if iscrowd != 0:
            continue
        
        if class_name not in target_classes:
            continue

        x, y, w, h = map(int, ann["bbox"])

        # 🔴 фильтр размера
        if w < MIN_SIZE or h < MIN_SIZE:
            continue

        # создаём маску
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        segmentation = ann.get("segmentation", None)
        if isinstance(segmentation, list) and len(segmentation) > 0:
            for seg in segmentation:
                if len(seg) < 6:
                    continue
                pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [pts], 255)
        else:
            mask[y:y+h, x:x+w] = 255
        
        # проверка связности
        if not is_connected(mask):
            print('pizdec')
            continue  # объект состоит из нескольких частей → пропускаем

        segmentation = ann.get("segmentation", None)

        if isinstance(segmentation, list):
            # polygon
            for seg in segmentation:
                if len(seg) < 6:
                    continue
                pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [pts], 255)

        elif isinstance(segmentation, dict):
            try:
                from pycocotools import mask as maskUtils
        
                rle = segmentation
        
                #  если counts — список → конвертируем
                if isinstance(rle["counts"], list):
                    rle = maskUtils.frPyObjects(rle, rle["size"][0], rle["size"][1])
        
                decoded_mask = maskUtils.decode(rle)
        
                # если несколько масок → объединяем
                if len(decoded_mask.shape) == 3:
                    decoded_mask = np.any(decoded_mask, axis=2).astype(np.uint8)
        
                mask = np.maximum(mask, decoded_mask * 255)
        
            except ImportError:
                print("⚠️ pip install pycocotools")
                mask[y:y+h, x:x+w] = 255
        else:
            mask[y:y+h, x:x+w] = 255

        # вырезаем объект
        obj = cv2.bitwise_and(img, img, mask=mask)
        obj_cropped = obj[y:y+h, x:x+w]

        # имя файла
        idx = class_counters[class_name]
        out_name = f"{class_name}_{idx}.png"
        class_counters[class_name] += 1

        # путь с папкой класса
        class_folder = os.path.join(output_folder, class_name)
        out_path = os.path.join(class_folder, out_name)

        cv2.imwrite(out_path, obj_cropped)

    print(f"Обработано: {file_name}")

print("✅ Готово!")

Блок аугментации

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
from PIL import Image
from skimage.exposure import match_histograms

In [ ]:
BACKGROUND_DIR = r"background"  # папка с фоном
OBJECTS_DIR = r"output"        # папка с сохраненными объектами


os.makedirs(r"images", exist_ok=True) # создаем папку под готовые изображения
os.makedirs(r"CSVs", exist_ok=True)   # папку под CSV аннотации

OUTPUT_DIR = r"images"
CSV_PATH = r"CSVs"
UNIQUE_NAME = os.path.basename(BACKGROUND_DIR)

# число готовых изображений
NUMBER_OF_OUTPUT = 250

# среднее для пуассона
LAMBDA = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

backgrounds = [os.path.join(BACKGROUND_DIR, f) for f in os.listdir(BACKGROUND_DIR)]

In [ ]:
def remove_background(img: np.ndarray, threshold: int = 10) -> np.ndarray:
    """
    Удаляет фон (почти черный) и создает аккуратный alpha-канал.

    Параметры:
        img: ndarray (BGR)
        threshold: порог "насколько чёрный считается фоном"

    Возвращает:
        RGBA ndarray (с альфа-каналом)
    """

    if img is None:
        raise ValueError("Input image is None")

    # если есть альфа — убираем
    if img.shape[-1] == 4:
        img = img[:, :, :3]

    # конвертируем в серый для маски
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # создаем бинарную маску объекта
    mask = np.where(gray > threshold, 255, 0).astype(np.uint8)

    # морфологические операции для очистки шума и закрытия дыр
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # мягкие края через размытие
    mask = cv2.GaussianBlur(mask, (5, 5), 0)

    # нормализуем маску для применения к BGR
    mask_f = mask.astype(np.float32) / 255.0
    img_f = img.astype(np.float32)

    # применяем маску
    result_rgb = (img_f * mask_f[:, :, None]).astype(np.uint8)

    # создаем альфа-канал
    alpha = mask

    # объединяем в RGBA
    result = cv2.cvtColor(result_rgb, cv2.COLOR_BGR2BGRA)
    result[:, :, 3] = alpha

    return result

In [ ]:
from skimage.exposure import match_histograms

def shift_colors(img, bg_patch=None):
    """
    Выравнивает цвет объекта под фон через histogram matching.

    img: ndarray (H, W, 3)
    bg_patch: ndarray (H, W, 3) — кусок фона (желательно того же размера)

    return: ndarray
    """

    # если нет фона — fallback на лёгкий random shift
    if bg_patch is None:
        shift = np.random.randint(-10, 10, 3)
        img = img.astype(np.int16)
        for i in range(3):
            img[:, :, i] += shift[i]
        return np.clip(img, 0, 255).astype(np.uint8)

    # приводим к uint8
    img = img.astype(np.uint8)
    bg_patch = bg_patch.astype(np.uint8)

    # если размеры не совпадают — ресайзим фон
    if bg_patch.shape[:2] != img.shape[:2]:
        bg_patch = cv2.resize(bg_patch, (img.shape[1], img.shape[0]))

    try:
        matched = match_histograms(img, bg_patch, channel_axis=-1)
        matched = np.clip(matched, 0, 255).astype(np.uint8)
        return matched
    except Exception:
        # fallback если что-то пошло не так
        return img

In [ ]:
def blur_edges(alpha):
    """
    Размывает только границы маски (реалистично)
    
    alpha: ndarray (0–1 или 0–255)
    return: ndarray (того же формата)
    """

    # нормализация в 0–1
    alpha = alpha.astype(np.float32)
    if alpha.max() > 1.0:
        alpha = alpha / 255.0

    # бинарная маска
    binary = (alpha > 0.5).astype(np.uint8)

    # находим границу (edge band)
    kernel = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(binary, kernel, iterations=1)
    eroded = cv2.erode(binary, kernel, iterations=1)

    edge = (dilated - eroded).astype(np.float32)

    # размытая версия всей маски
    blurred = cv2.GaussianBlur(alpha, (15, 15), 0)

    # комбинируем:
    # внутри — оригинал, на границе — blur
    result = alpha * (1 - edge) + blurred * edge

    # возвращаем в исходный диапазон
    if alpha.max() <= 1.0:
        return result
    else:
        return (result * 255).astype(np.uint8)

In [ ]:
def select_four_points(image):
    """
    Кликни 4 точки на изображении.
    Возвращает np.array([...], dtype=np.int32) и печатает в готовом формате.
    """

    points = []
    img_copy = image.copy()

    def mouse_callback(event, x, y, flags, param):
        nonlocal points, img_copy

        if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
            points.append((x, y))

            # рисуем точку
            cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)

            # номер точки
            cv2.putText(img_copy, str(len(points)), (x+5, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.namedWindow("Select 4 points")
    cv2.setMouseCallback("Select 4 points", mouse_callback)

    while True:
        cv2.imshow("Select 4 points", img_copy)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('r'):  # сброс
            points = []
            img_copy = image.copy()

        if key == ord('q'):  # выход
            break

        if len(points) == 4:
            break

    cv2.destroyAllWindows()

    polygon = np.array(points, dtype=np.int32)

    return polygon

In [ ]:
def position(bg_h, bg_w, polygon):
    x_min, y_min = polygon.min(axis=0)
    x_max, y_max = polygon.max(axis=0)

    for _ in range(100):  # пробуем 100 раз
        x = random.randint(x_min, x_max)
        y = random.randint(y_min, y_max)

        if cv2.pointPolygonTest(polygon, (x, y), False) >= 0:
            return x, y

    # fallback (редко)
    return int((x_min + x_max) / 2), int((y_min + y_max) / 2)

In [ ]:
def get_scale(bg_w, x, y, polygon):
    """
    Возвращает масштаб объекта с учётом позиции (perspective scaling)
    """
    # ЗАДАЙ РАЗМЕРЫ
    scale_near = 0.1 * bg_w   # нижняя точка (ближе к камере)
    scale_far  = 0.01 * bg_w   # верхняя точка (дальше)

    # границы по Y полигона
    y_min = polygon[:,1].min()
    y_max = polygon[:,1].max()

    # нормализация позиции (0 = далеко, 1 = близко)
    t = (y - y_min) / (y_max - y_min + 1e-6)
    t = np.clip(t, 0, 1)

    # линейная интерполяция
    scale = scale_far + t * (scale_near - scale_far)

    return scale

In [ ]:
def paste_object(bg, obj, x, y):
    h, w = obj.shape[:2]

    alpha = obj[:, :, 3] / 255.0
    alpha = blur_edges(alpha)

    for c in range(3):
        bg[y:y+h, x:x+w, c] = (
            alpha * obj[:, :, c] +
            (1 - alpha) * bg[y:y+h, x:x+w, c]
        )

    return bg

In [ ]:
def bb_intersects(bb1, bb2):
    """
    Проверяет пересечение двух bbox.
    bb = (x_min, y_min, x_max, y_max)
    """
    x1_min, y1_min, x1_max, y1_max = bb1
    x2_min, y2_min, x2_max, y2_max = bb2

    # если один bbox полностью слева/справа/сверху/снизу от другого → пересечения нет
    if x1_max <= x2_min or x1_min >= x2_max:
        return False
    if y1_max <= y2_min or y1_min >= y2_max:
        return False
    return True

def check_collision(new_bb, existing_bbs):
    """
    Проверяет, пересекается ли new_bb с любым из existing_bbs
    """
    for bb in existing_bbs:
        if bb_intersects(new_bb, bb):
            return True
    return False

In [ ]:
# выбор четырехугольника для вставки объектов
img = cv2.imread(r"background.jpg") # путь к бэкграунду
polygon = select_four_points(img)
bg_path = r"background.jpg"


for i in range(NUMBER_OF_OUTPUT):
    bg = cv2.imread(bg_path)

    bg_h, bg_w = bg.shape[:2]

    num_objects = np.random.poisson(LAMBDA)
    num_objects = max(1, min(num_objects, 7)) 

    annotations = []
    existing_bbs = []  # сюда будем сохранять все уже вставленные bbox

    num_of_pasted_objects = 0
    for _ in range(num_objects):
        # ВЫБИРАЕМ РАНДОМНУЮ ПАПКУ ИЗ ПАПОК С КЛАССАМИ
        subfold = os.path.join(OBJECTS_DIR, random.choice(os.listdir(OBJECTS_DIR)))
        objects = [os.path.join(subfold, f) for f in os.listdir(subfold)]
        
        if len(objects) == 0:
            #print("No objects")
            continue
            
        obj_path = random.choice(objects)
        obj = cv2.imread(obj_path)
        if obj is None:
            #print("Skipped: object is None")
            continue  
        
        obj = remove_background(obj)
    
        # позиция
        x, y = position(bg_h, bg_w, polygon)
    
        # масштаб
        target_w = int(get_scale(bg_w, x, y, polygon))
        scale = target_w / obj.shape[1]
        target_h = int(obj.shape[0] * scale)

        if target_h >= bg_h or target_w >= bg_w or target_h < 75 or target_w < 75:
            #print("Too small/big")
            continue
    
        # проверяем пересечение с уже вставленными объектами
        new_bb = (x, y, x + target_w, y + target_h)
        
        if check_collision(new_bb, existing_bbs):
            #print("Skipped: collision")
            continue  # пересечение есть → пробуем другой объект / позицию
    
        obj = cv2.resize(obj, (target_w, target_h))
        obj[:, :, :3] = shift_colors(obj[:, :, :3])

        h, w = obj.shape[:2]

        # ограничиваем размеры, если объект выходит за границы
        h = min(h, bg_h - y)
        w = min(w, bg_w - x)
        
        if h <= 0 or w <= 0:
            continue
        
        # обрезаем объект и маску
        obj = obj[:h, :w]
        mask_patch = mask_patch[:h, :w]
        
        bg = paste_object(bg, obj, x, y)

        # добавляем в список существующих bbox
        existing_bbs.append(new_bb)
        num_of_pasted_objects += 1
    
        annotations.append({
            "class_name": os.path.basename(subfold),
            "x_min": x,
            "y_min": y,
            "x_max": x + target_w,
            "y_max": y + target_h
        })

    # сохранение
    if num_of_pasted_objects != 0:
        save_csv_annotation(
            os.path.join(CSV_PATH, f"img_{i}.csv"),
            f"img_{i}.jpg",
            annotations
        )
    
        out_path = os.path.join(OUTPUT_DIR, f"img_{i}.jpg")
        cv2.imwrite(out_path, bg)


Сохраняем в CSV

In [ ]:
import csv

def save_csv_annotation(csv_path, filename, annotations):
    """
    Сохраняет аннотации в CSV

    filename: имя изображения
    annotations: список dict:
        {
            "class_id": int,
            "x_min": int,
            "y_min": int,
            "x_max": int,
            "y_max": int
        }
    """

    with open(csv_path, mode='w', newline='') as f:
        writer = csv.writer(f)

        writer.writerow(["filename", "class_name", "x_min", "y_min", "x_max", "y_max"])

        for ann in annotations:
            writer.writerow([
                filename,
                ann["class_name"],
                ann["x_min"],
                ann["y_min"],
                ann["x_max"],
                ann["y_max"]
            ])

Проверка 


In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# === Папки с CSV и изображениями ===
CSV_DIR = CSV_PATH      # папка с CSV
IMAGES_DIR = OUTPUT_DIR # папка с изображениями

# === Функция для отрисовки боксов на изображении ===
def draw_boxes_on_image(image, df):
    """
    image: numpy array (BGR)
    df: DataFrame с колонками ["filename", "class_name", "x_min", "y_min", "x_max", "y_max"]
    """
    for _, row in df.iterrows():
        x_min = int(row["x_min"])
        y_min = int(row["y_min"])
        x_max = int(row["x_max"])
        y_max = int(row["y_max"])
        class_name = str(row["class_name"])

        color = (0, 255, 0)  # зеленый

        # Рисуем прямоугольник
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, 2)

        # Рисуем подпись
        (w, h), _ = cv2.getTextSize(class_name, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(image, (x_min, y_min - h - 5), (x_min + w, y_min), color, -1)
        cv2.putText(image, class_name, (x_min, y_min - 3), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 0, 0), 2, lineType=cv2.LINE_AA)
    return image

# === Основной цикл по CSV файлам ===
for csv_file in os.listdir(CSV_DIR):
    if not csv_file.endswith(".csv"):
        continue

    csv_path = os.path.join(CSV_DIR, csv_file)
    df = pd.read_csv(csv_path)

    # имя соответствующего изображения
    img_name = csv_file.replace(".csv", ".jpg")
    img_path = os.path.join(IMAGES_DIR, img_name)

    if not os.path.exists(img_path):
        print(f"Изображение не найдено: {img_path}")
        continue

    # читаем изображение
    img = cv2.imread(img_path)
    if img is None:
        print(f"Не удалось открыть: {img_path}")
        continue

    # рисуем боксы
    img = draw_boxes_on_image(img, df)

    # BGR → RGB для matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # отображаем
    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.title(img_name)
    plt.axis("off")
    plt.show()